# Information Extraction (Invoices Only)

Pipeline stage 2: extract structured fields from documents the classifier has already labelled as `invoice`.

**Target fields** (per assignment):
- `invoice_number`
- `invoice_date`
- `due_date`
- `issuer`
- `recipient`
- `total`

## Input
We do **not** use `data/labeled/dataset.csv` here — the classification preprocessing lowercases text and strips numbers/dates, which destroys everything IE needs. Instead we read the raw SROIE box files directly.

## SROIE ground-truth coverage
SROIE labels only 4 of our 6 fields: `company` (→ issuer), `date` (→ invoice_date), `address`, `total`. It has **no** labels for `invoice_number`, `due_date`, or `recipient`. For those we'll rely on rules and validate on supplementary real invoice PDFs collected for the live demo.

## Public API
```python
extract_invoice_fields(text: str) -> dict  # always returns all 6 keys, missing → None
```

## 0. Imports & paths

In [39]:
import os
import re
import json
from pathlib import Path

import pandas as pd

if Path.cwd().name == 'notebooks':
    os.chdir('..')

import sys
sys.path.insert(0, str(Path.cwd()))

SROIE_ROOT = Path('data/raw/invoices/SROIE2019')
BOX_DIRS = [SROIE_ROOT / 'train' / 'box', SROIE_ROOT / 'test' / 'box']
GT_DIRS  = [SROIE_ROOT / 'train' / 'entities', SROIE_ROOT / 'test' / 'entities',
            SROIE_ROOT / 'train' / 'key',       SROIE_ROOT / 'test' / 'key']

print('cwd:', Path.cwd())
print('SROIE exists:', SROIE_ROOT.exists())

cwd: /Users/ryanmuenker/Desktop/School/STATISTICAL LEARNING/group project/Document-Classification-and-Information-Extraction-
SROIE exists: True


## 1. Load raw SROIE text (preserving case, numbers, punctuation)

Each box file is `x1,y1,x2,y2,x3,y3,x4,y4,text`. We keep original casing and join tokens in file order, which approximately follows reading order. We also keep the filename as an ID so we can join with ground-truth labels.

In [40]:
def load_sroie_box(path: Path) -> str:
    lines = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) >= 9:
                token = ','.join(parts[8:]).strip()
                if token:
                    lines.append(token)
    return '\n'.join(lines)


def load_all_sroie() -> pd.DataFrame:
    rows = []
    for box_dir in BOX_DIRS:
        if not box_dir.exists():
            continue
        for fp in sorted(box_dir.glob('*.txt')):
            rows.append({'doc_id': fp.stem, 'split': box_dir.parent.name, 'text': load_sroie_box(fp)})
    return pd.DataFrame(rows)


df_inv = load_all_sroie()
print(f'Loaded {len(df_inv)} SROIE invoices')
if len(df_inv):
    print('\n--- sample ---')
    print(df_inv.iloc[0]['text'][:500])

Loaded 973 SROIE invoices

--- sample ---
TAN WOON YANN
BOOK TA .K(TAMAN DAYA) SDN BND
789417-W
NO.53 55,57 & 59, JALAN SAGU 18,
TAMAN DAYA,
81100 JOHOR BAHRU,
JOHOR.
DOCUMENT NO : TD01167104
DATE:
25/12/2018 8:13:39 PM
CASHIER:
MANIS
MEMBER:
CASH BILL
CODE/DESC
PRICE
DISC
AMOUNT
QTY
RM
RM
9556939040116
KF MODELLING CLAY KIDDY FISH
1 PC
*
9.000
0.00
9.00
TOTAL:
ROUR DING ADJUSTMENT:
0.00
ROUND D TOTAL (RM):
9.00
CASH
10.00
CHANGE
1.00
GOODS SOLD ARE NOT RETURNABLE OR
EXCHANGEABLE
***
***
THANK YOU
PLEASE COME AGAIN !
9.00


## 2. Load SROIE ground truth (for evaluation)

Labels live in per-document JSON files with keys `company`, `date`, `address`, `total`. We map them to our schema.

In [41]:
def load_sroie_ground_truth() -> pd.DataFrame:
    rows = []
    for gt_dir in GT_DIRS:
        if not gt_dir.exists():
            continue
        for fp in sorted(gt_dir.glob('*.txt')) + sorted(gt_dir.glob('*.json')):
            try:
                with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
                    gt = json.load(f)
            except Exception:
                continue
            rows.append({
                'doc_id': fp.stem,
                'gt_issuer': gt.get('company'),
                'gt_invoice_date': gt.get('date'),
                'gt_address': gt.get('address'),
                'gt_total': gt.get('total'),
            })
    return pd.DataFrame(rows)


df_gt = load_sroie_ground_truth()
print(f'Loaded {len(df_gt)} ground-truth records')
df_gt.head()

Loaded 973 ground-truth records


,doc_id,gt_issuer,gt_invoice_date,gt_address,gt_total
0,X00016469612,BOOK TA .K (TAMAN DAYA) SDN BHD,25/12/2018,"NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 8...",9.00
1,X00016469619,INDAH GIFT & HOME DECO,19/10/2018,"27, JALAN DEDAP 13, TAMAN JOHOR JAYA, 81100 JO...",60.30
2,X00016469620,MR D.I.Y. (JOHOR) SDN BHD,12-01-19,"LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERI...",33.90
3,X00016469622,YONGFATT ENTERPRISE,25/12/2018,NO 122.124. JALAN DEDAP 13 81100 JOHOR BAHRU,80.90
4,X00016469623,MR D.I.Y. (M) SDN BHD,18-11-18,"LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERI...",30.90


## 3. Rule-based extractors

Imported from `src/information_extraction.py` — the single source of truth.  
Editing the extractors there automatically updates both this notebook and the live service.

In [42]:
from src.information_extraction import (
    extract_invoice_fields,
    extract_invoice_date,
    extract_due_date,
    extract_invoice_number,
    extract_total,
    extract_issuer,
    extract_recipient,
    InvoiceFields,
)

# quick smoke test on one doc
if len(df_inv):
    sample = df_inv.iloc[0]['text']
    print(extract_invoice_fields(sample))


{'invoice_number': None, 'invoice_date': '25/12/2018', 'due_date': None, 'issuer': 'BOOK TA .K(TAMAN DAYA) SDN BND', 'recipient': None, 'total': '9.00'}


## 4. Run extractor on the full SROIE set

In [43]:
preds = df_inv['text'].apply(extract_invoice_fields).apply(pd.Series)
df_pred = pd.concat([df_inv[['doc_id']], preds], axis=1)
df_pred.head()

,doc_id,invoice_number,invoice_date,due_date,issuer,recipient,total
0,X00016469612,None,25/12/2018,None,BOOK TA .K(TAMAN DAYA) SDN BND,None,9.00
1,X00016469619,None,19/10/2018,None,TAN WOON YANN,None,60.31
2,X00016469620,None,12-01-19,None,MR D.T.Y. (JOHOR) SDN BHD,None,33.90
3,X00016469622,None,25/12/2018,None,YONGFATT ENTERPRISE,None,80.91
4,X00016469623,None,18-11-18,None,MR D.I.Y. (M) SDN BHD,None,30.90


## 5. Evaluate against SROIE ground truth

SROIE only covers `issuer`, `invoice_date`, `total`. We compute field-level exact-match and a relaxed normalized match.

In [44]:
def _norm(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ''
    return re.sub(r'[^a-z0-9]', '', str(s).lower())


def evaluate(df_pred: pd.DataFrame, df_gt: pd.DataFrame) -> pd.DataFrame:
    if df_gt.empty:
        print('No ground truth found — skipping eval.')
        return pd.DataFrame()
    df = df_pred.merge(df_gt, on='doc_id', how='inner')
    checks = {
        'issuer':       ('issuer',       'gt_issuer'),
        'invoice_date': ('invoice_date', 'gt_invoice_date'),
        'total':        ('total',        'gt_total'),
    }
    results = []
    for field, (pcol, gcol) in checks.items():
        p = df[pcol].map(_norm)
        g = df[gcol].map(_norm)
        have_gt = g != ''
        exact = (p == g) & have_gt
        substr = pd.Series(
            [(a in b) or (b in a) if a and b else False for a, b in zip(p, g)],
            index=df.index,
        )
        relaxed = have_gt & (p != '') & substr
        results.append({
            'field': field,
            'n_with_gt': int(have_gt.sum()),
            'exact_match': round(exact.sum() / max(have_gt.sum(), 1), 3),
            'relaxed_match': round(relaxed.sum() / max(have_gt.sum(), 1), 3),
        })
    return pd.DataFrame(results)


evaluate(df_pred, df_gt)


,field,n_with_gt,exact_match,relaxed_match
0,issuer,973,0.615,0.901
1,invoice_date,973,0.959,0.960
2,total,972,0.644,0.710


## 6. Error analysis

Print a handful of mismatches per field so we know what to improve in the regexes.

In [45]:
def show_mistakes(df_pred, df_gt, field, gt_col, n=5):
    if df_gt.empty:
        return
    df = df_pred.merge(df_gt, on='doc_id', how='inner')
    mask = df[field].map(_norm) != df[gt_col].map(_norm)
    for _, row in df[mask].head(n).iterrows():
        print(f"[{row['doc_id']}] pred={row[field]!r}  gt={row[gt_col]!r}")
    print()

show_mistakes(df_pred, df_gt, 'issuer', 'gt_issuer')
show_mistakes(df_pred, df_gt, 'invoice_date', 'gt_invoice_date')
show_mistakes(df_pred, df_gt, 'total', 'gt_total')

[X00016469612] pred='BOOK TA .K(TAMAN DAYA) SDN BND'  gt='BOOK TA .K (TAMAN DAYA) SDN BHD'
[X00016469619] pred='TAN WOON YANN'  gt='INDAH GIFT & HOME DECO'
[X00016469620] pred='MR D.T.Y. (JOHOR) SDN BHD'  gt='MR D.I.Y. (JOHOR) SDN BHD'
[X51005230617] pred='GOLDEN ARCHES RESTAURANTS SDN BHD'  gt='GERBANG ALAF RESTAURANTS SDN BHD'
[X51005268200] pred='ENTERPRISE (SETIA'  gt='AIK HUAT HARDWARE ENTERPRISE (SETIA ALAM) SDN BHD'

[X51005268400] pred='2017-12-28'  gt='12/28/2017'
[X51005447850] pred='04/03/2018'  gt='20180304'
[X51005568880] pred='5/40/160'  gt='19-09-17'
[X51005715010] pred=None  gt='25032018'
[X51005757349] pred=None  gt='24-MAR-2018'

[X00016469619] pred='60.31'  gt='60.30'
[X00016469622] pred='80.91'  gt='80.90'
[X51005230617] pred='11.00'  gt='26.60'
[X51005301661] pred=None  gt='73.00'
[X51005303661] pred=None  gt='73.00'



## 7. Demo path — PDF in, fields out

For the live demo the input is a PDF. This cell wires the upcoming `pdf_loader.py` (text-first, OCR fallback) into the extractor.

In [31]:
from src.pdf_loader import pdf_to_text

# Replace with a real invoice PDF path to test end-to-end
# pdf_path = "path/to/invoice.pdf"
# text = pdf_to_text(pdf_path)
# fields = extract_invoice_fields(text)
# print(fields)

# --- dry run on a SROIE box-file text (no PDF needed) ---
if len(df_inv):
    sample_text = df_inv.iloc[0]['text']
    fields = extract_invoice_fields(sample_text)
    print("doc_id :", df_inv.iloc[0]['doc_id'])
    for k, v in fields.items():
        print(f"  {k:<16}: {v}")

## Status

| Item | Status |
|---|---|
| `src/information_extraction.py` — production extractor | ✅ done |
| `src/pdf_loader.py` — pdfplumber + OCR fallback | ✅ done |
| `src/preprocessing.py` — `clean_for_classifier()` | ✅ done |
| `src/service.py` — FastAPI `/classify` endpoint | ✅ done |
| Date patterns: DD-MON-YYYY, DDMMYYYY, false-positive guard | ✅ done |
| Issuer: skip doc-type titles, stop at recipient marker | ✅ done |
| `due_date` / `recipient` — unscored (no SROIE labels) | validate on real PDFs |